# Разметка и подготовка данных: SRE Triage Assistant

Лабораторная работа 6. Плотников Максим Александрович, магистратура, 3 семестр.

Набор синтетических инцидентов телеком-оператора, версия 1.1. В ноутбуке собраны пять
пунктов задания: инструкция разметчика, версия набора, статистика разметки, проверка
согласия и код подготовки признаков. Расчёты лежат в `src/`, здесь только вызовы и
объяснения, поэтому числа в ноутбуке и в документах не расходятся.

Что внутри:

1. инструкция разметчика: какие метки ставим и по каким правилам;
2. версия набора 1.1: что изменилось и почему;
3. листы пилота: 30 карточек и 10 повторных, слепая разметка;
4. проверка согласия: каппа между проходами, сверка с набором, журнал расхождений;
5. альтернативный контроль: воспроизводимая демонстрация расчёта;
6. код предобработки: словарь и IDF только по train;
7. ограничения, которые важно не потерять.

Как воспроизвести: данные в репозиторий не коммитятся, они пересобираются из seed.
В Colab первая ячейка клонирует репозиторий и прогоняет пайплайн, если файлов нет.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/aldentecoffe/Triage-Assistant.git"

ROOT = Path.cwd()
if not (ROOT / "src" / "agreement.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "agreement.py").exists():
    REPO_DIR = Path.cwd() / "Triage-Assistant"
    if not (REPO_DIR / "src" / "agreement.py").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    ROOT = REPO_DIR

import os

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Каталог репозитория:", ROOT.name)

PIPELINE = ("src/generate_incidents.py", "src/prepare_dataset.py", "src/split_dataset.py")
if not (ROOT / "data" / "processed" / "incidents_train_v1.1.csv").exists():
    for script in PIPELINE:
        subprocess.run([sys.executable, script], check=True)

import pandas as pd
from IPython.display import Markdown, display

import agreement
import agreement_demo
import prepare_features
from incident_taxonomy import IMPACT_SEVERITY, ROOT_CAUSES, SCENARIOS, SERVICES, SEVERITIES
from split_dataset import find_violations

pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 120)
print("Готово: данные на месте")

Каталог репозитория: project


Готово: данные на месте


## 1. Как читать ноутбук

| Пункт задания | Где смотреть |
|---|---|
| Инструкция разметчика | раздел 2 и `docs/lab6/annotation_guideline.md` |
| Версия набора | раздел 3 и `docs/lab6/data_card_update.md` |
| Статистика разметки | раздел 4: объём, распределение ответов, уверенность |
| Проверка согласия | раздел 5: каппа, сверка с набором, журнал; `src/agreement.py` |
| Код предобработки | раздел 6: `src/prepare_features.py` |

Альтернативный контроль (раздел 6 в тексте отчёта) — демонстрация расчёта на
синтетических метках, когда второго разметчика нет.

## 2. Инструкция разметчика

Размечаем три метки по описанию карточки. Критичность ставится по последствию, а не по
симптому и не по значению метрики.

| Метка | Когда |
|---|---|
| `P1` | голос и регистрация в сети, платежи или магистраль не работают, обходного пути нет. Заказы, процессинг и интеграции сюда не входят |
| `P2` | функция не выполняется для части; остановлен заказ или интеграция; клиент видит ошибку, неверный счёт или нулевой баланс; операция не завершается по таймауту |
| `P3` | медленно, но завершается; очередь; часть ассортимента; обход есть: резерв, повтор, ручное оформление, 3G вместо VoLTE |
| `P4` | клиент не затронут: сверка, отчёты, справочники, мониторинг, самовосстановление. Если клиент видит ошибку — это не `P4` |
| `спорный` | правило не даёт ответа; в комментарии два возможных чтения |

Сервис определяется по системе, в которой отказ: `transport`, `platform`, `database`,
`ims`, `billing`, `bss`, `external`. Причина — по тому, что сломалось: `change`,
`capacity`, `hardware`, `software`, `network`, `external_dep`.

Полный текст с примерами и границами: `docs/lab6/annotation_guideline.md`, версия 1.1.

In [ ]:
rows = []
for scenario in SCENARIOS:
    marks = IMPACT_SEVERITY[scenario["id"]]
    for phrase, label in zip(scenario["impacts_ru"], marks["ru"]):
        rows.append({"severity": label, "impact": phrase})
    rows.append({"severity": marks["en"], "impact": scenario["impact_en"]})

impacts = pd.DataFrame(rows)
print("Последствий в каталоге:", len(impacts))
print(impacts["severity"].value_counts().sort_index().to_string())
print()
print("Примеры по классам:")
print(impacts.groupby("severity").head(2)[["severity", "impact"]].to_string(index=False))

Последствий в каталоге: 84
P1     7
P2    33
P3    35
P4     9

Примеры по классам:
severity                                                                           impact
      P3 около 400 префиксов ушло в резерв, задержка на магистральном направлении выросла
      P3                   часть клиентских префиксов анонсируется через резервный аплинк
      P2            новые префиксы не устанавливаются, часть трафика уходит в чёрную дыру
      P2                                   маршруты до части площадок пропадают на минуты
      P4                                  нагрузка перераспределилась на оставшиеся хосты
      P4                               метрики части узлов пропали, алерты не срабатывают
      P1                  абоненты в зоне обслуживания не могут зарегистрироваться в сети
      P1                   subscribers in the serving area cannot register in the network


## 3. Версия набора 1.1

Что изменилось по сравнению с 1.0 и почему:

- критичность больше не разыгрывается: класс выводится из формулировки последствия по
  таблице `IMPACT_SEVERITY`, поэтому метку можно объяснить и проверить;
- сценарии перераспределены по выборкам так, чтобы каждый класс критичности встречался
  в train, validation и test, а все шесть категорий причин — в train и test. В прежней
  раскладке в test не было ни одной карточки `P1`;
- у сценариев с причиной `software` убраны формулировки «новый билд» и «после обновления»:
  они не отличались от `change`, и граница классов была нерешаемой.

Тексты всех сценариев пересчитаны: генератор детерминирован по seed, но правка каталога
меняет поток случайных чисел. Сравнивать тексты версий построчно нельзя, сравниваются
инварианты.

In [ ]:
frames = {
    split: pd.read_csv(ROOT / "data" / "processed" / f"incidents_{split}_v1.1.csv", dtype=str, keep_default_na=False)
    for split in ("train", "validation", "test")
}
frame = pd.concat(frames.values(), ignore_index=True)

summary = pd.DataFrame(
    [
        {
            "выборка": split,
            "карточек": len(part),
            "сценариев": part["scenario_id"].nunique(),
            "событий": part["event_id"].nunique(),
            "сервисов": part["service"].nunique(),
            "причин": part["root_cause"].nunique(),
        }
        for split, part in frames.items()
    ]
)
print(summary.to_string(index=False))
print()
print("Всего:", len(frame), "карточек,", frame["event_id"].nunique(), "событий")
print("Дубликатов text_hash:", int(frame["text_hash"].duplicated().sum()))

shares = frame.groupby("split")["severity"].value_counts(normalize=True).mul(100).round(1).unstack()
print()
print("Доли критичности по выборкам, %:")
print(shares[list(SEVERITIES)].to_string())

   выборка  карточек  сценариев  событий  сервисов  причин
     train      3600         14     2479         7       6
validation      1200          7      796         7       5
      test      1200          7      822         7       6

Всего: 6000 карточек, 4097 событий
Дубликатов text_hash: 0

Доли критичности по выборкам, %:
severity      P1    P2    P3    P4
split                             
test        14.2  31.1  43.3  11.3
train        4.2  35.9  48.3  11.7
validation   8.0  46.6  36.8   8.6


Все четыре класса критичности есть в каждой выборке, в train и в test присутствуют все
шесть категорий причин, в validation нет `network` — это известное ограничение, унаследованное
от раскладки 1.0 и записанное в проверках ЛР 5. `P1` редкий: 417 карточек, 7,0 %, по
выборкам доля расходится из-за группового разделения по сценариям.

In [ ]:
violations = find_violations(frame)
print("Нарушений разделения:", len(violations) or "нет")

events = frame.groupby("event_id").size()
print("Событий из двух и больше карточек:", int((events > 1).sum()))
print("Карточек с соседом по событию:", int(events[events > 1].sum()))
print("Каскадных пар:", 4)

Нарушений разделения: нет
Событий из двух и больше карточек: 1245
Карточек с соседом по событию: 3148
Каскадных пар: 4


## 4. Листы разметки

Пилот: 30 карточек из 21 сценария и 10 из них повторно, через сутки. Первый лист
`data/annotation/pass_a.csv`, второй — `pass_b.csv`, соответствие номеров — `key.csv`.
В листах только номер карточки и описание: метки, `scenario_id`, `event_id`, сервис и
причина туда не попадают, иначе проверялась бы память о сгенерированных метках.

In [ ]:
first = agreement.collect(agreement.read_sheet(ROOT / "data" / "annotation" / "pass_a.csv"))
second = agreement.collect(agreement.read_sheet(ROOT / "data" / "annotation" / "pass_b.csv"))
key = agreement.read_key(ROOT / "data" / "annotation" / "key.csv")

print("Первый проход:", len(first), "карточек;", "повторный:", len(second))
print("Колонки листа:", list(pd.read_csv(ROOT / "data" / "annotation" / "pass_a.csv", nrows=0).columns))
print()
print("Распределение ответов первого прохода:")
for column in ("severity", "service", "root_cause"):
    counts = first[f"{column}_label"].value_counts().to_dict()
    print(f"  {column}: {counts}")
print("Уверенность:", first["confidence"].value_counts().to_dict())
print("Спорных:", {c: int(first[f"{c}_state"].eq('unsure').sum()) for c in ("severity", "service", "root_cause")})

Первый проход: 30 карточек; повторный: 10
Колонки листа: ['card', 'description', 'severity', 'service', 'root_cause', 'confidence', 'comment']

Распределение ответов первого прохода:
  severity: {'P2': 10, 'P3': 9, 'P4': 6, 'P1': 5}
  service: {'billing': 8, 'platform': 7, 'database': 5, 'transport': 3, 'ims': 3, 'bss': 2, 'external': 2}
  root_cause: {'capacity': 10, 'software': 8, 'network': 4, 'hardware': 3, 'external_dep': 3, 'change': 2}
Уверенность: {'3': 26, '2': 4}
Спорных: {'severity': 0, 'service': 0, 'root_cause': 0}


## 5. Проверка согласия

Две таблицы. Первая — первый проход против второго, только по 10 карточкам, которые
размечались дважды: она отвечает, устойчива ли инструкция. Вторая — каждый проход
против меток набора: первый проход целиком (30 карточек), второй — свои 10.

Пороги выбраны до начала: каппа ниже 0,4 — инструкция не готова; 0,4–0,6 — переписываются
границы классов; выше 0,6 — инструкция годится для пилота.

In [ ]:
gold = pd.read_csv(
    ROOT / "data" / "interim" / "incidents_clean_v1.1.csv", dtype=str, keep_default_na=False
)[["incident_id", *agreement.ALLOWED]]

combined = agreement.combine(first, second, key, gold)
display(Markdown(agreement.report(combined)))

log = agreement.discrepancies(combined)
print("Журнал расхождений:", len(log), "строк")
print(log["label"].value_counts().to_string())

### Согласие первого и второго прохода

| Метка | Сравнено | Совпало | Доля совпадений | Каппа Коэна | Квадратичная каппа |
|---|---|---|---|---|---|
| Критичность | 10 | 6 | 0,600 | 0,481 | 0,556 |
| Сервис | 10 | 7 | 0,700 | 0,634 | — |
| Категория причины | 10 | 6 | 0,600 | 0,444 | — |

### Сверка с метками набора

| Метка | Проход A: сравнено | Проход A: доля | Проход B: сравнено | Проход B: доля |
|---|---|---|---|---|
| Критичность | 30 | 0,367 | 10 | 0,400 |
| Сервис | 30 | 0,667 | 10 | 0,900 |
| Категория причины | 30 | 0,500 | 10 | 0,600 |

Спорных решений в первом проходе: Критичность — 0, Сервис — 0, Категория причины — 0

Журнал расхождений: 47 строк
Критичность          20
Категория причины    17
Сервис               10


Что видно: согласие проходов 0,600 по критичности, 0,700 по сервису и 0,600 по причине
(каппа 0,481 / 0,634 / 0,444). С набором совпадение ниже: 0,367 / 0,667 / 0,500.

Главное — критичность. Расхождения не случайны и не объясняются одной ошибкой: одинаковые
по смыслу последствия получили разные метки. «Оформление заказов остановлено» размечено
как `P1`, а «Order processing is stopped» — как `P4`, хотя по правилу это один и тот же
`P2`. Заказы и процессинг уходили в `P1`, внутренняя отчётность — в `P2` и `P3`.

Причина процедурная: в форме была памятка из четырёх строк, а не таблица решений. Поэтому
инструкция переведена в версию 1.1: добавлены явные исключения для `P1` и `P4`, правило
«завершается — `P3`, не завершается — `P2`», требование одинаковой метки для одинакового
смысла на русском и английском, уточнены границы сервисов и причин. Полная таблица теперь
стоит прямо в форме разметки.

In [ ]:
sample = log.head(8).copy()
sample["первый"] = sample["first"].fillna("—")
sample["второй"] = sample["second"].fillna("—")
sample["набор"] = sample["dataset"].fillna("—")
print(sample[["card", "label", "первый", "второй", "набор"]].to_string(index=False))

card             label       первый       второй        набор
A001 Категория причины     hardware            —     capacity
A002       Критичность           P1            —           P2
A002            Сервис      billing            —          bss
A002 Категория причины     software            —     hardware
A003       Критичность           P4           P3           P3
A003 Категория причины external_dep     capacity external_dep
A004 Категория причины      network            —     hardware
A005 Категория причины external_dep external_dep     capacity


## 6. Альтернативный контроль

Второго разметчика нет, поэтому кроме отложенного повтора есть воспроизводимая
демонстрация расчёта: 120 карточек, проход A — метки набора, проход B — они же с
детерминированными изменениями. Это проверка кода, а не разметка: числа ниже ничего не
говорят о качестве правила, и в метрики проекта не входят.

`src/agreement_demo.py` печатает то же самое из командной строки.

In [ ]:
first_demo, second_demo, key_demo, gold_demo = agreement_demo.build_sheets(
    str(ROOT / "data" / "processed" / "incidents_validation_v1.1.csv"), 120
)
demo_combined = agreement.combine(first_demo, second_demo, key_demo, gold_demo)
display(Markdown(agreement.report(demo_combined)))
print("Демонстрационных расхождений:", len(agreement.discrepancies(demo_combined)))

### Согласие первого и второго прохода

| Метка | Сравнено | Совпало | Доля совпадений | Каппа Коэна | Квадратичная каппа |
|---|---|---|---|---|---|
| Критичность | 120 | 106 | 0,883 | 0,812 | 0,889 |
| Сервис | 120 | 115 | 0,958 | 0,951 | — |
| Категория причины | 120 | 116 | 0,967 | 0,957 | — |

### Сверка с метками набора

| Метка | Проход A: сравнено | Проход A: доля | Проход B: сравнено | Проход B: доля |
|---|---|---|---|---|
| Критичность | 120 | 1,000 | 120 | 0,883 |
| Сервис | 120 | 1,000 | 120 | 0,958 |
| Категория причины | 120 | 1,000 | 120 | 0,967 |

Спорных решений в первом проходе: Критичность — 0, Сервис — 0, Категория причины — 0

Демонстрационных расхождений: 23


## 7. Код предобработки

`src/prepare_features.py`: словарь и IDF считаются только по train, к validation и test
применяются неизменные веса. Параметры зафиксированы до обучения модели: `ngram_range=(1, 2)`,
`min_df=2`, `max_features=20000`, `sublinear_tf`. Признак один — `description`.

Проверка утечки: если заменить все описания в validation и test на новые слова, словарь
и IDF обязаны остаться прежними.

In [ ]:
splits = prepare_features.load_splits(
    {
        "train": str(ROOT / "data" / "processed" / "incidents_train_v1.1.csv"),
        "validation": str(ROOT / "data" / "processed" / "incidents_validation_v1.1.csv"),
        "test": str(ROOT / "data" / "processed" / "incidents_test_v1.1.csv"),
    }
)
vectorizer, manifest = prepare_features.build(splits)
print("Словарь:", manifest["vocabulary_size"], "признаков")
print("Отпечаток словаря:", manifest["vocabulary_sha256"])
print("Строки:", manifest["rows"])

tampered = {name: part.copy() for name, part in splits.items()}
tampered["validation"]["description"] = "совсем новые слова про космос"
tampered["test"]["description"] = "другие новые слова про океан"
other, _ = prepare_features.build(tampered)

same_vocab = list(vectorizer.get_feature_names_out()) == list(other.get_feature_names_out())
same_idf = bool((vectorizer.idf_ == other.idf_).all())
print()
print("Словарь не изменился:", same_vocab)
print("IDF не изменился:", same_idf)

Словарь: 3799 признаков
Отпечаток словаря: 44c848662e62332db0c179052fa1b8846e3fccb10e0da4c78dcfc601f228574e
Строки: {'train': 3600, 'validation': 1200, 'test': 1200}

Словарь не изменился: True
IDF не изменился: True


Словарь и веса не двигаются от правок validation и test: подготовка не использует test.
Параметры и отпечаток словаря записаны в `docs/lab6/features_manifest.json`, поэтому
перед обучением модели видно, что состав признаков не менялся.

В модель идёт только `description`: `title` повторяет слова описания у 100 % карточек,
`scenario_id` связан с метками на 1,00, дециль `affected_ci` — на 0,62 с сервисом,
поэтому эти поля остаются журнальными.

## 8. Ограничения

1. Согласие с метками набора низкое по критичности (0,367): правило из инструкции 1.0
   человек не воспроизводит. Значит F1 модели по критичности будет измерять
   воспроизводимость таблицы `IMPACT_SEVERITY`, а не качество триажа на реальном потоке.
2. Повторных карточек 10 — каппа индикативна, доверительный интервал шире значения.
3. Разметчик один, автор проекта; независимого второго нет. Отложенный повтор измеряет
   устойчивость решения, а не его истинность.
4. После версии 1.1 переразметка не выполнялась, эффект правок не измерен.
5. Набор синтетический; метки набора тоже авторские, расхождение с ними не значит, что
   разметчик ошибся.

## 8.1 Проверки

Полный набор тестов запускается отдельно: `python -m pytest -q` — 255 тестов. Они проверяют
расчёт согласия (включая квадратичную каппу и журнал), подготовку признаков (словарь и IDF
только по train), генерацию слепых листов и правило критичности. Внутри ноутбука тесты
не запускаются: окружение Colab отличается от локального, и отчёт о прогоне не должен
зависеть от версии pandas.

## 9. Карта файлов

| Файл | Что внутри |
|---|---|
| `docs/lab6/annotation_guideline.md` | инструкция разметчика 1.1: правила, примеры, границы, журнал, версии |
| `docs/lab6/annotation_results.md` | таблицы согласия, сверка с набором, разбор расхождений |
| `docs/lab6/pilot_plan.md` | протокол пилота: выборка, слепота, отложенный повтор, пороги |
| `docs/lab6/data_card_update.md` | Data Card 1.2: версия набора, состав, ограничения |
| `docs/lab6/features_manifest.json` | отпечаток подготовки признаков |
| `data/annotation/` | листы `pass_a.csv`, `pass_b.csv`, ключ, формы |
| `src/prepare_features.py` | TF-IDF только по train |
| `src/agreement.py` | расчёт согласия и журнал |
| `src/agreement_demo.py` | демонстрация расчёта на синтетических метках |
| `src/annotation_sheet.py`, `src/generate_incidents.py`, `src/prepare_dataset.py`, `src/split_dataset.py` | сборка набора и слепых листов |
| `tests/test_agreement.py`, `test_features.py`, `test_annotation.py`, `test_severity_rule.py` | проверки расчёта и подготовки |

Репозиторий: https://github.com/aldentecoffe/Triage-Assistant